# Using a New Machine Learning Classifier

In this activity, you’ll evaluate how our earlier trading strategy behaves when it uses a different machine learning classification model.

## Instructions:

1. Run all the cells up to the “Add a New Machine Learning Model” section.

2. Import the `LogisticRegression` model from scikit-learn.

    > **Rewind** Recall that `LogisticRegression` models are used for binary classification problems.

3. Using the same training data that the SVM model used (`X_train_scaled` and `y_train`), fit the `LogisticRegression` model.

4. Use the trained model to predict the trading signals for the training data. Use the `classification_report` module to evaluate the model.

5. Backtest the `LogisticRegression` model to evaluate its performance.

6. Compare the performance of the logistic regression and SVM models using the classification reports generated with the testing data.  Did the logistic regression model perform better than SVM?


## References:

[SKLearn SVM - SVC Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

[SKLearn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

### Read the CSV file into Pandas DataFrame

In [5]:
# Import the OHLCV dataset into a Pandas Dataframe
trading_df = pd.read_csv(
    Path("14.3/01_Using_Machine_Learning/Resources/ohlcv.csv"), 
    index_col="date", 
    infer_datetime_format=True, 
)

# Review the DataFrame
trading_df.head()

/var/folders/s9/ml6qrgdx03zdn76qyj8422mh0000gn/T/ipykernel_2437/2765480066.py:2: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trading_df = pd.read_csv(


,open,high,low,close,volume
date,,,,,
10/19/18 9:30,16.90,17.18,16.90,17.095,11522
10/19/18 9:45,17.11,17.44,17.11,17.400,70593
10/19/18 10:00,17.40,17.40,17.25,17.280,38885
10/19/18 10:15,17.27,17.27,17.18,17.200,37046
10/19/18 10:30,17.21,17.37,17.19,17.200,46874


### Add a daily return values column to the DataFrame

In [7]:
# Calculate the daily returns using the closing prices and the pct_change function
trading_df['actual_returns'] = trading_df['close'].pct_change()

# Drop all NaN values from the DataFrame
trading_df = trading_df.dropna()

# Review the DataFrame
display(trading_df.head())
display(trading_df.tail())

,open,high,low,close,volume,actual_returns
date,,,,,,
10/19/18 9:45,17.11,17.44,17.11,17.40,70593,0.017841
10/19/18 10:00,17.40,17.40,17.25,17.28,38885,-0.006897
10/19/18 10:15,17.27,17.27,17.18,17.20,37046,-0.004630
10/19/18 10:30,17.21,17.37,17.19,17.20,46874,0.000000
10/19/18 10:45,17.20,17.20,17.10,17.12,11266,-0.004651


,open,high,low,close,volume,actual_returns
date,,,,,,
9/4/20 14:45,6.225,6.26,6.220,6.250,55512,0.003210
9/4/20 15:00,6.255,6.27,6.245,6.250,65810,0.000000
9/4/20 15:15,6.250,6.29,6.250,6.275,202630,0.004000
9/4/20 15:30,6.270,6.28,6.250,6.255,130140,-0.003187
9/4/20 15:45,6.250,6.28,6.250,6.250,190278,-0.000799


### Generating the Features and Target Sets

In [8]:
# Define a window size of 4
short_window = 4

# Create a simple moving average (SMA) using the short_window and assign this to a new columns called sma_fast
trading_df['sma_fast'] = trading_df['close'].rolling(window=short_window).mean()
trading_df.tail()

,open,high,low,close,volume,actual_returns,sma_fast
date,,,,,,,
9/4/20 14:45,6.225,6.26,6.220,6.250,55512,0.003210,6.22875
9/4/20 15:00,6.255,6.27,6.245,6.250,65810,0.000000,6.23875
9/4/20 15:15,6.250,6.29,6.250,6.275,202630,0.004000,6.25125
9/4/20 15:30,6.270,6.28,6.250,6.255,130140,-0.003187,6.25750
9/4/20 15:45,6.250,6.28,6.250,6.250,190278,-0.000799,6.25750


In [9]:
# Define a window size of 100
long_window = 100

# Create a simple moving average (SMA) using the long_window and assign this to a new columns called sma_slow
trading_df['sma_slow'] = trading_df['close'].rolling(window=long_window).mean()
trading_df.tail()


,open,high,low,close,volume,actual_returns,sma_fast,sma_slow
date,,,,,,,,
9/4/20 14:45,6.225,6.26,6.220,6.250,55512,0.003210,6.22875,6.27030
9/4/20 15:00,6.255,6.27,6.245,6.250,65810,0.000000,6.23875,6.26985
9/4/20 15:15,6.250,6.29,6.250,6.275,202630,0.004000,6.25125,6.26910
9/4/20 15:30,6.270,6.28,6.250,6.255,130140,-0.003187,6.25750,6.26855
9/4/20 15:45,6.250,6.28,6.250,6.250,190278,-0.000799,6.25750,6.26785


In [10]:
# Drop the NaNs using dropna()
trading_df = trading_df.dropna()

#### Create the features set

In [11]:
# Assign a copy of the sma_fast and sma_slow columns to a new DataFrame called X
X = trading_df[['sma_fast','sma_slow']].copy()

# Display sample data
display(X.head())
display(X.tail())

,sma_fast,sma_slow
date,,
10/24/18 15:00,15.65250,16.3403
10/24/18 15:15,15.61875,16.3216
10/24/18 15:30,15.55375,16.3029
10/24/18 15:45,15.47625,16.2844
10/25/18 9:30,15.40250,16.2656


,sma_fast,sma_slow
date,,
9/4/20 14:45,6.22875,6.27030
9/4/20 15:00,6.23875,6.26985
9/4/20 15:15,6.25125,6.26910
9/4/20 15:30,6.25750,6.26855
9/4/20 15:45,6.25750,6.26785


#### Create the target set

In [13]:
# Create a new column in the trading_df called signal setting its value to zero.
trading_df['signal'] = 0.0

In [15]:
# Create the signal to buy
trading_df.loc[(trading_df['actual_returns']>= 0), 'signal'] = 1

In [17]:
# Create the signal to sell
trading_df.loc[(trading_df['actual_returns'] < 0), 'signal'] = -1

In [18]:
trading_df.tail()

,open,high,low,close,volume,actual_returns,sma_fast,sma_slow,signal
date,,,,,,,,,
9/4/20 14:45,6.225,6.26,6.220,6.250,55512,0.003210,6.22875,6.27030,1.0
9/4/20 15:00,6.255,6.27,6.245,6.250,65810,0.000000,6.23875,6.26985,1.0
9/4/20 15:15,6.250,6.29,6.250,6.275,202630,0.004000,6.25125,6.26910,1.0
9/4/20 15:30,6.270,6.28,6.250,6.255,130140,-0.003187,6.25750,6.26855,-1.0
9/4/20 15:45,6.250,6.28,6.250,6.250,190278,-0.000799,6.25750,6.26785,-1.0


### Split the Data Into Training and Testing Datasets

#### Creating the Training Datasets

In [19]:
# Imports 
from pandas.tseries.offsets import DateOffset

In [20]:
# Select the start of the training period
training_begin = X.index.min()

# Display the training begin date
print(training_begin)

1/10/19 10:00


In [22]:
# Select the ending period for the training data with an offset of 3 months
#training_end = X.index.min() + DateOffset(months=3)
training_end = X.index.min() + pd.Timedelta(days=90)
# Display the training end date
print(training_end)

TypeError: can only concatenate str (not "Timedelta") to str